# SRA „Virtual Neuron“ Emergence Validation Experiment v2

In diesem Notizbuch trainieren wir gleichzeitig ein SRA-Modell für **25 Aufgaben in 5 Domänen × 5 Aufgaben**, wobei jede Domäne eine grundlegend andere Zeichenverteilung aufweist.

- **Englischer Text (NL)** – a-z, Leerzeichen, Satzzeichen
- **Python-Code (CODE)** – def, :, =, (, #, Einrückung
- **Mathematische Ausdrücke (MATH)** – Ziffern, +, -, *, /, =
- **DNA-Sequenzen (DNA)** – nur A, T, G, C
- **CSV-Daten (CSV)** – Ziffern, Kommas

Da die Eingabe jeder Domäne völlig unterschiedliche Zeichentypen und -verteilungen aufweist, überprüfen wir, ob der Router **die Aufgabe ohne Hinweise wie eine Aufgaben-ID selbstständig identifizieren kann** und unterschiedliche Gruppen von Synapsen (= virtuellen Neuronen) pro Domäne bilden kann.

In [ ]:
import sys
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
import random
import numpy as np
import collections

if 'google.colab' in sys.modules:
    !git clone https://github.com/JunSuzukiJapan/SynapticRouter.git
    %cd SynapticRouter
    !pip install torch matplotlib seaborn numpy nbformat

sys.path.append('.')
sys.path.append('./src')
if 'google.colab' not in sys.modules:
    sys.path.append('..')
    sys.path.append('../src')

from src.sra_gpu_models import MoESRAModel
from src.sra_experiment import make_optimizer, load_balance_loss, specialization_loss

device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. Aufgabendesign und Datengenerator

Wir definieren 5 Domänen × 5 Aufgaben. Jede Domäne hat eine grundlegend andere Eingabezeichenverteilung.
Das Modell arbeitet auf Zeichenebene (ASCII) und kodiert den Eingabetext vor der Verarbeitung über Zeichen → ASCII-Ganzzahl.

In [ ]:
import random, string

# ===== Character-Level Encoder =====
VOCAB_SIZE = 128  # ASCII
PAD = 0
BOS = 1
EOS = 2

def encode(text):
    """ASCII string -> int list (with BOS/EOS)"""
    return [BOS] + [ord(c) for c in text] + [EOS]

def decode(ids):
    """int list -> string (control tokens excluded)"""
    return ''.join(chr(i) for i in ids if i >= 32)

def pad_to(seq, length):
    return seq[:length] + [PAD] * max(0, length - len(seq))

# ===== Domain 1: English Text (NL) =====
WORDS = ["hello", "world", "python", "learn", "great", "smart", "open", "code",
         "data", "fast", "model", "brain", "build", "train", "green", "black"]

def nl_upper(w=None):
    w = w or random.choice(WORDS)
    return w, w.upper()

def nl_vowel_count(w=None):
    w = w or random.choice(WORDS)
    c = sum(1 for ch in w if ch in 'aeiou')
    return w, str(c)

def nl_reverse(w=None):
    w = w or random.choice(WORDS)
    return w, w[::-1]

def nl_length(w=None):
    w = w or random.choice(WORDS)
    return w, str(len(w))

def nl_first_char(w=None):
    w = w or random.choice(WORDS)
    return w, w[0]

# ===== Domain 2: Python Code (CODE) =====
VARS = ["x", "y", "z", "n", "val", "res", "cnt", "idx"]
OPS  = ["+", "-", "*"]

def code_indent(v=None, op=None, n=None):
    v = v or random.choice(VARS); op = op or random.choice(OPS); n = n or random.randint(1,9)
    line = f"return {v} {op} {n}"
    return line, "    " + line

def code_is_comment():
    if random.random() < 0.5:
        line = f"# {random.choice(['add values','init','loop','check'])}"
        return line, "yes"
    else:
        v = random.choice(VARS); n = random.randint(1,9)
        return f"{v} = {n}", "no"

def code_operator(v=None, op=None, n=None):
    v = v or random.choice(VARS); op = op or random.choice(OPS); n = n or random.randint(1,9)
    return f"{v} = a {op} {n}", op

def code_varname(v=None, n=None):
    v = v or random.choice(VARS); n = n or random.randint(1,99)
    return f"{v} = {n}", v

def code_has_return():
    if random.random() < 0.5:
        v = random.choice(VARS)
        return f"return {v}", "yes"
    else:
        v = random.choice(VARS); n = random.randint(1,9)
        return f"{v} = {n}", "no"

# ===== Domain 3: Math Expressions (MATH) =====
def math_add():
    a, b = random.randint(1,19), random.randint(1,19)
    return f"{a}+{b}=", str(a+b)

def math_sub():
    a, b = random.randint(1,19), random.randint(1,19)
    lo, hi = min(a,b), max(a,b)
    return f"{hi}-{lo}=", str(hi-lo)

def math_mul():
    a, b = random.randint(1,9), random.randint(1,9)
    return f"{a}*{b}=", str(a*b)

def math_gt():
    a, b = random.randint(1,19), random.randint(1,19)
    return f"{a}>{b}?", "yes" if a > b else "no"

def math_lt():
    a, b = random.randint(1,19), random.randint(1,19)
    return f"{a}<{b}?", "yes" if a < b else "no"

# ===== Domain 4: DNA Sequences (DNA) =====
BASES = ['A', 'T', 'G', 'C']
COMP  = {'A':'T', 'T':'A', 'G':'C', 'C':'G'}

def rand_dna(n=5):
    return ''.join(random.choices(BASES, k=n))

def dna_complement():
    s = rand_dna()
    return s, ''.join(COMP[c] for c in s)

def dna_reverse():
    s = rand_dna()
    return s, s[::-1]

def dna_repeat():
    s = rand_dna(3)
    return s, s * 2

def dna_count_g():
    s = rand_dna()
    return s, str(s.count('G'))

def dna_has_a():
    s = rand_dna()
    return s, "yes" if 'A' in s else "no"

# ===== Domain 5: CSV Data (CSV) =====
def rand_csv(n=4):
    nums = [random.randint(1, 20) for _ in range(n)]
    return ','.join(str(x) for x in nums), nums

def csv_count():
    s, nums = rand_csv()
    return s, str(len(nums))

def csv_max():
    s, nums = rand_csv()
    return s, str(max(nums))

def csv_min():
    s, nums = rand_csv()
    return s, str(min(nums))

def csv_first():
    s, nums = rand_csv()
    return s, str(nums[0])

def csv_last():
    s, nums = rand_csv()
    return s, str(nums[-1])

# ===== Task Registry =====
def make_sample(task_fn):
    result = task_fn()
    return result  # (input_str, output_str)

ALL_TASKS = {
    # NL
    "nl_upper":       lambda: nl_upper(),
    "nl_vowel_count": lambda: nl_vowel_count(),
    "nl_reverse":     lambda: nl_reverse(),
    "nl_length":      lambda: nl_length(),
    "nl_first_char":  lambda: nl_first_char(),
    # CODE
    "code_indent":    lambda: code_indent(),
    "code_is_comment":lambda: code_is_comment(),
    "code_operator":  lambda: code_operator(),
    "code_varname":   lambda: code_varname(),
    "code_has_return":lambda: code_has_return(),
    # MATH
    "math_add":       lambda: math_add(),
    "math_sub":       lambda: math_sub(),
    "math_mul":       lambda: math_mul(),
    "math_gt":        lambda: math_gt(),
    "math_lt":        lambda: math_lt(),
    # DNA
    "dna_complement": lambda: dna_complement(),
    "dna_reverse":    lambda: dna_reverse(),
    "dna_repeat":     lambda: dna_repeat(),
    "dna_count_g":    lambda: dna_count_g(),
    "dna_has_a":      lambda: dna_has_a(),
    # CSV
    "csv_count":      lambda: csv_count(),
    "csv_max":        lambda: csv_max(),
    "csv_min":        lambda: csv_min(),
    "csv_first":      lambda: csv_first(),
    "csv_last":       lambda: csv_last(),
}

DOMAIN_COLORS = {
    "nl":   "#4C72B0",
    "code": "#DD8452",
    "math": "#55A868",
    "dna":  "#C44E52",
    "csv":  "#8172B3",
}

def get_domain(task_name):
    return task_name.split('_')[0]

print(f"Total tasks: {len(ALL_TASKS)}")
for domain in ['nl','code','math','dna','csv']:
    tasks = [t for t in ALL_TASKS if get_domain(t)==domain]
    print(f"  {domain}: {tasks}")

# Sanity check
for name, fn in list(ALL_TASKS.items())[:5]:
    inp, out = fn()
    print(f"[{name}] '{inp}' -> '{out}'")


## 2. Modellaufbau und Schulung

Initialisieren Sie ein SRA-Modell auf Zeichenebene (ASCII 128) und trainieren Sie es für alle 25 Aufgaben gleichzeitig.
Da die Zeichenverteilungen in den Eingabetexten grundsätzlich unterschiedlich sind, sollte der Router jede Aufgabe ohne Aufgaben-ID autonom identifizieren und an verschiedene Synapsen weiterleiten.

In [ ]:
# Model configuration
MAX_SEQ_LEN = 24   # Max input character length (input + output)
VOCAB_SIZE   = 128 # ASCII

DIM          = 64
LAYERS       = 2
NUM_SYNAPSES = 32
K            = 2
SYN_HIDDEN   = 128

model = MoESRAModel(
    vocab_size   = VOCAB_SIZE,
    dim          = DIM,
    layers       = LAYERS,
    num_synapses = NUM_SYNAPSES,
    k            = K,
    syn_hidden   = SYN_HIDDEN,
).to(device)
optimizer = make_optimizer(model, lr=1e-3)
total_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total_params:,}")
print(f"Vocab size: {VOCAB_SIZE} (ASCII character-level)")
print(f"Synapses: {NUM_SYNAPSES}, k={K}")

def make_multitask_batch(tasks, batch_size, max_len=MAX_SEQ_LEN):
    """Sample uniformly from each task, encode the text, and form a batch."""
    xs, ys = [], []
    task_names = list(tasks.keys())
    for _ in range(batch_size):
        task_name = random.choice(task_names)
        inp_str, out_str = tasks[task_name]()
        x = pad_to(encode(inp_str), max_len)
        y = pad_to(encode(out_str), max_len)
        xs.append(x)
        ys.append(y)
    x = torch.tensor(xs, dtype=torch.long, device=device)
    y = torch.tensor(ys, dtype=torch.long, device=device)
    return x, y


In [ ]:
# Training loop
print("Multitask Training on 25 Tasks started...")
model.train()

epochs     = 2000
batch_size = 128

for epoch in range(epochs):
    x, y = make_multitask_batch(ALL_TASKS, batch_size)
    
    optimizer.zero_grad()
    y_in = torch.cat([torch.full((y.size(0), 1), BOS, dtype=torch.long, device=device), y[:, :-1]], dim=1)
    outputs, routing_weights, _ = model(x, y_in)
    
    loss    = F.cross_entropy(outputs.reshape(-1, VOCAB_SIZE), y.reshape(-1), ignore_index=PAD)
    lb_loss = load_balance_loss(routing_weights) * 0.1
    total_loss = loss + lb_loss
    
    total_loss.backward()
    optimizer.step()
    
    if (epoch + 1) % 200 == 0:
        print(f"Epoch {epoch+1}/{epochs} | Loss: {loss.item():.4f} | LB Loss: {lb_loss.item():.4f}")

print("Training finished!")


## 3. Visualisierung und Analyse virtueller Neuronen

Nach dem Training extrahieren wir, welche Synapsen der Router für jede Aufgabe auswählt. Indem wir diese nach Domänen gruppieren und als Heatmap darstellen, überprüfen wir, ob die Synapsennutzung gemeinsame Muster aufweist (d. h. die Entstehung virtueller Neuronen).

In [ ]:
def get_task_routing_vector(task_name, task_fn, samples=20):
    """Compute the actual synapse usage frequency for a text task.
    Exclude PAD tokens, and return the fraction of valid tokens that selected each synapse in top-k."""
    model.eval()
    synapse_counts = torch.zeros(NUM_SYNAPSES, device=device)
    total_valid_tokens = 0
    
    with torch.no_grad():
        for _ in range(samples):
            inp_str, out_str = task_fn()
            x       = torch.tensor([pad_to(encode(inp_str), MAX_SEQ_LEN)], dtype=torch.long, device=device)
            y_dummy = torch.tensor([pad_to(encode(out_str), MAX_SEQ_LEN)], dtype=torch.long, device=device)
            y_in    = torch.cat([torch.full((1, 1), BOS, dtype=torch.long, device=device), y_dummy[:, :-1]], dim=1)
            
            # Mask of valid tokens (non-PAD)
            seq = torch.cat([x, y_in], dim=1)
            valid_mask = (seq != PAD) # (1, T)
            
            _, routing_logits, _ = model(x, y_in)
            logits = routing_logits[-1] # (1, T, N)
            
            # Get top-k indices
            _, topk_idx = torch.topk(logits, K, dim=-1)   # (1, T, K)
            
            # Keep only valid tokens
            valid_topk = topk_idx[valid_mask] # (V, K)
            
            # Count usage per synapse
            for k_idx in range(K):
                synapse_counts.index_add_(0, valid_topk[:, k_idx], torch.ones(valid_topk.size(0), device=device))
                
            total_valid_tokens += valid_topk.size(0)

    # Fraction of valid tokens for which each synapse was selected in top-k (max 1.0)
    synapse_freq = synapse_counts / total_valid_tokens
    return synapse_freq.cpu().numpy()

# Collect synapse usage frequencies for all tasks
task_vectors = {}
for task_name, task_fn in ALL_TASKS.items():
    task_vectors[task_name] = get_task_routing_vector(task_name, task_fn)

# Tasks ordered by domain
ordered_tasks = (
    [t for t in ALL_TASKS if get_domain(t)=="nl"]   +
    [t for t in ALL_TASKS if get_domain(t)=="code"] +
    [t for t in ALL_TASKS if get_domain(t)=="math"] +
    [t for t in ALL_TASKS if get_domain(t)=="dna"]  +
    [t for t in ALL_TASKS if get_domain(t)=="csv"]
)

heatmap_data = np.array([task_vectors[t] for t in ordered_tasks])

fig, (ax_color, ax) = plt.subplots(1, 2, figsize=(16, 10),
                                    gridspec_kw={"width_ratios": [0.04, 1]})

import matplotlib.colors as mcolors, matplotlib.patches as mpatches

color_matrix = np.zeros((len(ordered_tasks), 1, 3))
for i, t in enumerate(ordered_tasks):
    color_matrix[i, 0] = mcolors.to_rgb(DOMAIN_COLORS[get_domain(t)])

ax_color.imshow(color_matrix, aspect="auto")
ax_color.set_xticks([])
ax_color.set_yticks(range(len(ordered_tasks)))
ax_color.set_yticklabels(ordered_tasks, fontsize=8)

# Draw the frequency heatmap
sns.heatmap(heatmap_data, ax=ax, cmap="Blues",
            xticklabels=range(NUM_SYNAPSES),
            yticklabels=ordered_tasks, linewidths=0.3)
ax.set_xlabel("Synapse Index", fontsize=12)
ax.set_title("Active Synapses Frequency Heatmap (top-k) / 5 Domains x 5 Tasks", fontsize=13)
ax.set_yticklabels(ordered_tasks, fontsize=8)

legend_patches = [mpatches.Patch(color=c, label=d.upper()) for d, c in DOMAIN_COLORS.items()]
ax.legend(handles=legend_patches, loc="upper right", fontsize=9)

plt.tight_layout()
plt.savefig("virtual_neuron_heatmap_v2.png", dpi=150, bbox_inches="tight")
plt.show()
print("Heatmap saved.")


In [ ]:
# Virtual neuron clustering extraction and hierarchical analysis

import networkx as nx

# 1. Extract Universal Core Synapses shared across the whole brain
# Synapses used by 40% (10 tasks) or more out of all 25 tasks are treated specially as foundational parts
UNIVERSAL_THRESHOLD = int(len(ALL_TASKS) * 0.4)

synapse_task_count = collections.defaultdict(int)
for task, vector in task_vectors.items():
    active_indices = np.where(vector >= 0.15)[0]
    for syn in active_indices:
        synapse_task_count[syn] += 1

universal_cores = set(syn for syn, count in synapse_task_count.items() if count >= UNIVERSAL_THRESHOLD)

# 2. Get the specialized synapses for each task
task_specialized = {}
for task, vector in task_vectors.items():
    active_indices = set(np.where(vector >= 0.15)[0])
    # Task-specific synapses, excluding the Universal Cores
    task_specialized[task] = active_indices - universal_cores

# 3. Build a graph network using Jaccard similarity
G = nx.Graph()
for task in ALL_TASKS.keys():
    G.add_node(task)

tasks = list(ALL_TASKS.keys())
for i in range(len(tasks)):
    for j in range(i + 1, len(tasks)):
        set_a = task_specialized[tasks[i]]
        set_b = task_specialized[tasks[j]]
        
        # If neither task has specialized synapses, treat similarity as 0
        if not set_a and not set_b:
            continue
            
        intersection = len(set_a & set_b)
        union = len(set_a | set_b)
        jaccard = intersection / union if union > 0 else 0
        
        # Connect with an edge (same virtual neuron) if similarity >= 40% (0.4)
        if jaccard >= 0.4:
            G.add_edge(tasks[i], tasks[j])

# 4. Extract connected components (virtual neurons) and print the hierarchy
connected_components = list(nx.connected_components(G))
# Sort by size, largest first
connected_components.sort(key=len, reverse=True)

print("=== Extracted Virtual Neurons (Cell Assemblies) Hierarchy ===\n")

for i, comp in enumerate(connected_components, 1):
    comp_tasks = list(comp)
    
    # Collect synapses used by all tasks in this group
    group_synapses = collections.defaultdict(int)
    for task in comp_tasks:
        active_indices = np.where(task_vectors[task] >= 0.15)[0]
        for syn in active_indices:
            group_synapses[syn] += 1
            
    # Universal: belongs to Universal Cores AND used in this group
    used_universals = set(syn for syn in group_synapses.keys() if syn in universal_cores)
    
    # Assembly Core: not Universal, used by at least half of this group's tasks
    half_size = max(1, len(comp_tasks) / 2)
    assembly_cores = set(syn for syn, count in group_synapses.items() 
                         if syn not in universal_cores and count >= half_size)
                         
    # Peripheral: specialized synapses other than Assembly Cores
    peripherals = set(syn for syn in group_synapses.keys() 
                      if syn not in universal_cores and syn not in assembly_cores)
    
    print(f"[Virtual Neuron {i}] (Belonging Tasks: {len(comp_tasks)})")
    for t in sorted(comp_tasks):
        print(f"  - {t}")
    print(f"  |- Universal Cores : {[int(x) for x in sorted(list(used_universals))]}  (foundational parts shared across the whole brain)")
    print(f"  |- Assembly Cores  : {[int(x) for x in sorted(list(assembly_cores))]}  (primary parts of this functional group)")
    print(f"  +- Peripheral      : {[int(x) for x in sorted(list(peripherals))]}  (peripheral parts specific to some tasks)")
    print()


Wie oben gezeigt, können wir bestätigen, dass **Gruppen von Aufgaben, die „genau die gleiche Kombination von Synapsen“ verwenden, ein einziges unabhängiges „virtuelles Neuron“ innerhalb des Netzwerks bilden**. Ähnliche Aufgaben gehören zum selben virtuellen Neuron, während völlig unterschiedliche Aufgaben separate virtuelle Neuronen bilden.

### Diskussionspunkte
* **Gemeinsame vs. individuelle Synapsen**: Überprüfen Sie, ob es Synapsen gibt, die breit über viele Domänen (den gemeinsamen Kern) hinweg feuern, im Vergleich zu Synapsen, die nur für Aufgaben innerhalb derselben Domäne (spezialisiert/individuell) stark feuern.
* **Bildung virtueller Neuronen (Zellverbände)**: Die **kombinierten Feuermuster (Gruppierungen)** dieser „gemeinsamen Synapsen“ und „einzelnen Synapsen“ können als das „virtuelle Neuron“ betrachtet werden, das diese Domäne/Aufgabe ausführt.
* **Unabhängiges Auslösen**: Beobachten Sie bei Aufgaben mit grundlegend unterschiedlichen Regeln (z. B. einer Substitutionsverschlüsselung), ob sie einen unabhängigen Satz von Synapsenkombinationen verwenden, der sich von allen anderen Aufgaben unterscheidet.